# LLM-Driven Ontology Generation: Pipeline Demonstration

This notebook demonstrates the complete **Top-Down Skeleton + Bottom-Up Population** algorithm for generating RDF ontologies from a domain using LLMs.

We'll walk through all 4 phases:
1. **Structured Seed** — Generate a multi-level taxonomic skeleton
2. **Structural Validation** — Validate and prune the seed structure
3. **UCB1 Expansion** — Iteratively expand the ontology using a multi-armed bandit
4. **RDF Serialization** — Emit hierarchical RDF triples and visualize

## Setup

Configure the LLM client and ontology parameters.

In [ ]:
import json
import os
from ontogen import ChatGpt, Ontology, OntologyLevel, DEFAULT_LEVEL_SCHEMA

# Configuration
DOMAIN = "Star Trek"  # Change this to explore other domains
API_KEY = os.getenv("OPENAI_API_KEY", "")
MODEL = "gpt-4-turbo"
SEED_SIZE = 5  # Number of top-level classes to seed
MAX_ITERATIONS = 3  # Max expansion iterations
EXPLORATION_CONSTANT = 1.4  # UCB1 exploration parameter
SIMILARITY_THRESHOLD = 50  # Minimum similarity (0-100) to accept relationships

print(f"Domain: {DOMAIN}")
print(f"API Key configured: {bool(API_KEY)}")
print(f"Default level schema: {[level.name for level in DEFAULT_LEVEL_SCHEMA]}")

In [ ]:
# Initialize LLM client and ontology
agent = ChatGpt(api_key=API_KEY, model=MODEL)

ontology = Ontology(
    domain=DOMAIN,
    agent=agent,
    level_schema=DEFAULT_LEVEL_SCHEMA,
    exploration_constant=EXPLORATION_CONSTANT,
    max_iterations=MAX_ITERATIONS,
    similarity_threshold=SIMILARITY_THRESHOLD,
)

print(f"Ontology initialized for domain: {ontology.domain}")

## Phase 1: Structured Seed

Generate a multi-level taxonomic skeleton from the domain using the LLM.
The seed captures the initial structure with classes, subclasses, and instances.

In [ ]:
# Generate structured seed taxonomy
print(f"Generating {SEED_SIZE}-level taxonomic skeleton for '{DOMAIN}'...")
seed = ontology.generate_initial_terms(num_classes=SEED_SIZE)

if seed:
    print(f"\nSeed domain: {seed.get('domain')}")
    print(f"Number of top-level classes: {len(seed.get('taxonomy', []))}")
    print("\nRaw seed taxonomy (JSON):")
    print(json.dumps(seed, indent=2))
else:
    print("ERROR: Seed generation failed. Check API key and try again.")

## Phase 2: Structural Validation

Convert the seed to a graph and validate its structure using pairwise LLM similarity checks.
This phase prunes weak edges and identifies any orphaned nodes.

In [ ]:
# Create DiGraph from seed
print("Converting seed taxonomy to internal graph...")
ontology.create_seed_ontology()

if ontology.ontology_graph.number_of_nodes() > 0:
    print(f"Graph created: {ontology.ontology_graph.number_of_nodes()} nodes, {ontology.ontology_graph.number_of_edges()} edges")
    print(f"\nNodes by level:")
    for level in DEFAULT_LEVEL_SCHEMA:
        nodes_at_level = [n for n, d in ontology.ontology_graph.nodes(data=True) if d.get('level') == level.name]
        print(f"  {level.name}: {len(nodes_at_level)} nodes ({', '.join(nodes_at_level[:3])}{'...' if len(nodes_at_level) > 3 else ''})")
else:
    print("ERROR: Graph creation failed.")

In [ ]:
# Validate structure: check parent-child, sibling, and cross-branch relationships
print("\nValidating structure with pairwise similarity checks...")
validation_summary = ontology.validate_structure()

print("\nValidation Summary:")
for key, value in validation_summary.items():
    print(f"  {key}: {value}")

print(f"\nGraph after validation: {ontology.ontology_graph.number_of_nodes()} nodes, {ontology.ontology_graph.number_of_edges()} edges")

## Phase 3: UCB1 Expansion

Iteratively expand the ontology using a multi-armed bandit (UCB1) strategy.
For each iteration, we select the most promising expandable node and generate new candidates.

In [ ]:
# Run full generation pipeline (seed + validate + expand loop + serialize)
print(f"Running full pipeline with max_iterations={MAX_ITERATIONS}...")
print(f"Similarity threshold: {SIMILARITY_THRESHOLD}")
print(f"Exploration constant (UCB1): {EXPLORATION_CONSTANT}\n")

ontology.generate_ontology()

print(f"\nPipeline complete!")
print(f"Final graph: {ontology.ontology_graph.number_of_nodes()} nodes, {ontology.ontology_graph.number_of_edges()} edges")

In [ ]:
# Display expansion statistics
print("\nExpansion Statistics:")
print(f"Total nodes in final ontology: {ontology.ontology_graph.number_of_nodes()}")
print(f"Total edges in final ontology: {ontology.ontology_graph.number_of_edges()}")

# Count nodes by level
print(f"\nNodes by level:")
for level in DEFAULT_LEVEL_SCHEMA:
    nodes_at_level = [n for n, d in ontology.ontology_graph.nodes(data=True) if d.get('level') == level.name]
    print(f"  {level.name}: {len(nodes_at_level)} nodes")

# Show UCB1 bandit statistics
print(f"\nUCB1 Bandit Statistics (node selection):")
if hasattr(ontology, 'bandit_stats') and ontology.bandit_stats:
    for node, stats in sorted(ontology.bandit_stats.items(), key=lambda x: x[1]['n_visits'], reverse=True)[:5]:
        visits = stats.get('n_visits', 0)
        reward = stats.get('total_reward', 0.0)
        mean_reward = reward / visits if visits > 0 else 0
        print(f"  {node}: {visits} visits, avg reward = {mean_reward:.2f}")
else:
    print("  (Bandit statistics not available)")

## Phase 4: RDF Serialization

Convert the expanded ontology to RDF triples and serialize to standard formats (Turtle, XML, JSON-LD).

In [ ]:
# Build RDF graph from the ontology
print("Building RDF graph from ontology...")
ontology.build_ontology()

if ontology.rdf is not None:
    num_triples = len(ontology.rdf)
    print(f"RDF graph built: {num_triples} triples")
else:
    print("ERROR: RDF graph construction failed.")

In [ ]:
# Serialize to Turtle format
print("Serializing ontology to Turtle format...")
turtle_output = ontology.serialize_ontology(format="turtle")

# Display first 50 lines of Turtle output
print("\nTurtle Output (first 50 lines):")
lines = turtle_output.split('\n')
for line in lines[:50]:
    print(line)

if len(lines) > 50:
    print(f"... ({len(lines) - 50} more lines)")

In [ ]:
# Save to file
output_path = "../output/ontology.ttl"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, 'w') as f:
    f.write(turtle_output)

print(f"Ontology saved to: {output_path}")
print(f"File size: {os.path.getsize(output_path)} bytes")

## Visualization

Visualize the ontology structure both as an RDF graph and an internal DiGraph with level-based coloring.

In [ ]:
# Visualize the RDF graph
print("Rendering RDF graph (via rdf2dot/graphviz)...")
try:
    ontology.visualize()
except Exception as e:
    print(f"Note: RDF visualization requires graphviz. Error: {e}")

In [ ]:
# Visualize the internal DiGraph with level-based node coloring
print("Rendering internal DiGraph with level-based coloring...")
try:
    ontology.visualize_graph()
except Exception as e:
    print(f"Note: DiGraph visualization requires matplotlib. Error: {e}")

## Summary

The pipeline has successfully:
1. ✅ Generated a structured seed taxonomy with multiple levels
2. ✅ Validated the structure using pairwise LLM similarity checks
3. ✅ Expanded the ontology using UCB1-guided node selection
4. ✅ Serialized the result to RDF/Turtle format
5. ✅ Visualized the ontology structure

For detailed algorithm documentation and RDF schema information, see `docs/algorithm.md`.